# DDPM Experiment Workflow

This script collects the higher-level experiment functions from the original notebook:
`test_model` and `plot_model_comparison`, plus a sweep example.
These live here rather than in the `ddpm` package because they encode
a specific experiment workflow, not reusable library logic.

In [ ]:
import torch
import matplotlib.pyplot as plt

from ddpm import NoiseScheduler, UNet, train, find_lr, generate_image
from ddpm.dataset import load_mnist, get_noisy_loaders
from ddpm.utils import channel_list, model_name, path_name
from ddpm.viz import plot_model_comparison

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
scheduler = NoiseScheduler(T=1000)
train_set, test_set = load_mnist()
train_loader, test_loader = get_noisy_loaders(train_set, test_set, scheduler, batch_size=32)

## Workflow helpers

`test_model` is a convenience wrapper: find LR → train → generate.
Useful for quick comparisons without boilerplate.

In [ ]:
def test_model(model, train_loader, test_loader, epochs=5, weight_decay=1e-4,
               save_path='model.pkl', n_images=8):
    """
    Full pipeline: LR finder → train → generate.
    Returns (images, train_losses, test_losses).
    """
    suggested_lr = find_lr(model, train_loader)
    lr = suggested_lr * 0.5
    print(f"Recommended LR: {suggested_lr:.2e} → Used LR: {lr:.2e}")

    train_losses, test_losses = train(
        model, train_loader, test_loader,
        epochs=epochs,
        lr=lr,
        weight_decay=weight_decay,
        early_stopping_patience=10,
        save_path=save_path,
    )

    images = generate_image(model, scheduler, stochasticity=1.0, n_images=n_images)

    return images, train_losses, test_losses

## Model sweep

Run multiple architectures back to back and collect results for comparison.

In [ ]:
configs = [
    (64,  2),   # (channel0, convs_per_level)
    (128, 2),
    (64,  3),
]

model_names_list = []
image_list       = []
all_train_losses = []
all_test_losses  = []

for channel0, cpl in configs:
    print(f"\n=== {model_name(channel0, cpl)} ===")
    unet = UNet(channel_list(channel0), convs_per_level=cpl).to(device)
    imgs, train_losses, test_losses = test_model(
        unet, train_loader, test_loader,
        epochs=5,
        save_path=path_name(channel0, cpl),
        n_images=4,
    )
    model_names_list.append(model_name(channel0, cpl))
    image_list.append(imgs)
    all_train_losses.append(train_losses)
    all_test_losses.append(test_losses)

### Compare generated images across models

In [ ]:
plot_model_comparison(image_list, model_names_list)

### Compare loss curves across models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, tl, vl in zip(model_names_list, all_train_losses, all_test_losses):
    axes[0].plot(tl, label=name)
    axes[1].plot(vl, label=name)

axes[0].set_title('Train loss')
axes[1].set_title('Test loss')
for ax in axes:
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.legend()

plt.tight_layout()
plt.show()

### Generate and plot images for different stochasticities

In [1]:
def plot_stochasticities(model, scheduler: NoiseScheduler,
                         stochasticities=[0, 0.33, 0.67, 1.0], ncols=2):
    nrows = math.ceil(len(stochasticities) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 2, nrows * 2))

    for ax, s in zip(axes.flat, stochasticities):
        image = generate_image(model, scheduler, stochasticity=s, n_images=1)
        ax.imshow(image.cpu().detach().squeeze(), cmap='gray')
        ax.set_title(f's={s}')
        ax.axis('off')

    plt.tight_layout()
    plt.show()

NameError: name 'NoiseScheduler' is not defined

### Plot the outputs of different models efficiently

In [ ]:
def plot_model_comparison(images_list, model_names, ncol=None):
    for imgs, name in zip(images_list, model_names):
        imgs = imgs.detach().cpu()
        n_images = imgs.shape[0]
        current_ncol = ncol if ncol is not None else n_images
        nrow = math.ceil(n_images / current_ncol)

        if imgs.shape[1] in [1, 3]:
            imgs = imgs.permute(0, 2, 3, 1)
        if imgs.shape[-1] == 1:
            imgs = imgs.squeeze(-1)

        fig, axes = plt.subplots(nrow, current_ncol, figsize=(current_ncol * 2.5, nrow * 2.5))
        fig.suptitle(f"Model: {name}", fontsize=16, fontweight='bold', y=1.02)
        axes = axes.flatten() if n_images > 1 else [axes]

        for i in range(n_images):
            display_img = imgs[i]
            if display_img.min() < 0:
                display_img = (display_img + 1) / 2
            axes[i].imshow(display_img.clamp(0, 1), cmap='gray' if imgs.ndim == 3 else None)
            axes[i].axis('off')

        for j in range(n_images, len(axes)):
            axes[j].axis('off')

        plt.tight_layout()
        plt.show()